In [21]:
# importamos las librerías que necesitamos

# Tratamiento de datos
# -----------------------------------------------------------------------
import pandas as pd
import numpy as np
import warnings #Esto sirve para manejar las advertencias que puedan surgir durante la ejecución del código
warnings.filterwarnings('ignore')

# Imputación de nulos usando métodos avanzados estadísticos
# -----------------------------------------------------------------------
from sklearn.impute import SimpleImputer #para imputar con la media, mediana o moda
from sklearn.experimental import enable_iterative_imputer #para poder usar el IterativeImputer, que es un método de imputación más avanzado que el SimpleImputer
from sklearn.impute import IterativeImputer #para imputar con un modelo de regresión, es decir, que se entrena un modelo para predecir los valores nulos a partir de las demás variables
from sklearn.impute import KNNImputer #para imputar con el método de los k vecinos más cercanos, es decir, que se imputan los valores nulos con la media de los k vecinos más cercanos, donde la cercanía se mide en función de las demás variables

# Librerías de visualización
# -----------------------------------------------------------------------
import seaborn as sns
import matplotlib.pyplot as plt
# Configuración
# -----------------------------------------------------------------------
# Mostrar todas las columnas sin truncar
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


In [22]:
df = pd.read_csv('hr.csv')

print(f"El número de filas y columnas es: {df.shape[0]} filas y {df.shape[1]} columnas")

El número de filas y columnas es: 1474 filas y 35 columnas


In [23]:
df[df["JobSatisfaction"].isnull()]["Attrition"].value_counts()

Attrition
No     23
Yes     6
Name: count, dtype: int64

In [24]:
df.isnull().sum()

Age                          73
Attrition                     0
BusinessTravel              117
DailyRate                     0
Department                   29
DistanceFromHome              0
Education                     0
EducationField               58
EmployeeCount                 0
EmployeeNumber                0
EnvironmentSatisfaction       0
Gender                        0
HourlyRate                    0
JobInvolvement                0
JobLevel                      0
JobRole                       0
JobSatisfaction              29
MaritalStatus               132
MonthlyIncome                14
MonthlyRate                   0
NumCompaniesWorked            0
Over18                        0
OverTime                     44
PercentSalaryHike             0
PerformanceRating             0
RelationshipSatisfaction      0
StandardHours               164
StockOptionLevel              0
TotalWorkingYears             0
TrainingTimesLastYear        88
WorkLifeBalance               0
YearsAtC

In [25]:
#1 y 2 identificar valores con nulos
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print(nulos)

StandardHours            164
YearsWithCurrManager     148
MaritalStatus            132
BusinessTravel           117
TrainingTimesLastYear     88
Age                       73
EducationField            58
OverTime                  44
Department                29
JobSatisfaction           29
MonthlyIncome             14
dtype: int64


In [34]:

## Análisis detallado de valores nulos

#Calculamos exactamente cuántos nulos hay en cada columna y qué porcentaje representan sobre el total. Un porcentaje alto de nulos puede indicar que esa columna no es fiable.

nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': pct
}).sort_values('Nulos', ascending=False)

print("Columnas CON nulos:")
print(resumen_nulos[resumen_nulos['Nulos'] > 0].to_string())
print(f"\nColumnas SIN nulos: {(nulos == 0).sum()}")
print(f"Total de valores nulos en el dataset: {nulos.sum()}")

Columnas CON nulos:
                Nulos  Porcentaje (%)
StandardHours     164           11.13
BusinessTravel    117            7.94
OverTime           44            2.99
Department         29            1.97

Columnas SIN nulos: 31
Total de valores nulos en el dataset: 354


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1474 entries, 0 to 1473
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1474 non-null   float64
 1   Attrition                 1474 non-null   str    
 2   BusinessTravel            1357 non-null   str    
 3   DailyRate                 1474 non-null   int64  
 4   Department                1445 non-null   str    
 5   DistanceFromHome          1474 non-null   int64  
 6   Education                 1474 non-null   int64  
 7   EducationField            1474 non-null   str    
 8   EmployeeCount             1474 non-null   int64  
 9   EmployeeNumber            1474 non-null   int64  
 10  EnvironmentSatisfaction   1474 non-null   int64  
 11  Gender                    1474 non-null   str    
 12  HourlyRate                1474 non-null   int64  
 13  JobInvolvement            1474 non-null   int64  
 14  JobLevel           

In [29]:
columnas_mediana = ["Age", "YearsWithCurrManager", "TrainingTimesLastYear", "JobSatisfaction","MonthlyIncome"]
 
for col in columnas_mediana:
    df[col] = df[col].fillna(df[col].median())
 

In [32]:
df['MaritalStatus'] = df['MaritalStatus'].fillna(df['MaritalStatus'].mode()[0])

In [33]:
df['EducationField'] = df['EducationField'].fillna('Unknown')

In [ ]:
df['BusinessTravel'] = df['BusinessTravel'].fillna('Unknown')

In [ ]:
imputer_iter = IterativeImputer(max_iter = 100, random_state = 42)
df['Department'] = imputer_iter.fit_transform(df[['Department']])
 